In [2]:
import numpy as np 
import matplotlib.pyplot as plt
import torch 
import torchvision 
from torchvision import transforms, datasets
import os 
from PIL import Image
from collections import Counter
import random
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset
from sklearn.metrics import (roc_auc_score, roc_curve,
                             confusion_matrix, ConfusionMatrixDisplay,
                             f1_score, balanced_accuracy_score)

# ImageFolder stores labels in .targets
train_labels = torch.tensor(train_dataset.targets)
n_neg = (train_labels == 0).sum().item()  # Healthy
n_pos = (train_labels == 1).sum().item()  # OA

print(f"Train — Healthy: {n_neg}, OA: {n_pos}")
print(f"Val   — Healthy: {sum(1 for _, l in val_dataset if l == 0)}, "
      f"OA: {sum(1 for _, l in val_dataset if l == 1)}")
print(f"Test  — Healthy: {sum(1 for _, l in test_dataset if l == 0)}, "
      f"OA: {sum(1 for _, l in test_dataset if l == 1)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

# Up-weight the minority class in the loss
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float).to(device)
print(f"pos_weight: {pos_weight.item():.2f}")

In [ ]:
class Baseline(nn.Module):
    def __init__(self, dropout=0.5):
        super().__init__()

        # convolution part 
        self.conv = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # (1,128,128) -> (32,128,128)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                               # -> (32,64,64)

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # -> (64,64,64)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                               # -> (64,32,32)

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # -> (128,32,32)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),                               # -> (128,16,16)
        )

        # connected part 
        self.conn = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def forward(self,x):
        return self.conn(self.conv(x)).squeeze(1)
    
model = Baseline(dropout=0.5).to(device)
print(model) 
print(f"\nTrainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

def calculate_metrics(loader, model, threshold=0.5):
    model.eval() # turning the model in evaluation mode (important for dropout, batchnorm, ...) 
    all_probs, all_labels = [], []
    with torch.no_grad(): # disabling gradient calculation because we aren't updating the model 
        for X, y in loader:
            X = X.to(device)
            logits = model(X)
            probs  = torch.sigmoid(logits).cpu().numpy() # gives a percentage of how likely sick 
            all_probs.extend(probs)
            all_labels.extend(y.numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds  = (all_probs >= threshold).astype(int) # if the probability is above 50% we predict sick, otherwise healthy 

    return {
        "f1":       f1_score(all_labels, all_preds, zero_division=0),
        "bal_acc":  balanced_accuracy_score(all_labels, all_preds), 
        "auc":      roc_auc_score(all_labels, all_probs), # measures how well model separates two classes 
        "probs":    all_probs,
        "labels":   all_labels,
    }

In [ ]:
EPOCHS = 30 # an epoch is one complete pass of the training dataset through the neural network 
history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_bal_acc": []}

for epoch in range(1, EPOCHS + 1):
    # train, like a study session 
    model.train()
    train_loss = 0.0
    for X, y in train_loader: # the loader divides the big dataset into batches, otherwise too much data at once 
        X, y = X.to(device), y.to(device).float()
        optimizer.zero_grad()
        loss = criterion(model(X), y) # loss calculation with loss function above specified 
        loss.backward() # model works backwards from the error to see which neuron is responsible for error 
        optimizer.step() # adjusting of the weights 
        train_loss += loss.item() * len(y)
    train_loss /= len(train_dataset) # dividing by number of samples to get average per sample 

    # validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device).float()
            val_loss += criterion(model(X), y).item() * len(y)
    val_loss /= len(val_dataset)

    metrics = calculate_metrics(val_loader, model)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(metrics["f1"])
    history["val_bal_acc"].append(metrics["bal_acc"])

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"F1: {metrics['f1']:.3f} | BalAcc: {metrics['bal_acc']:.3f} | "
          f"AUC: {metrics['auc']:.3f}") 
    
    # overfitting is happening when train loss is decreasing but validation loss is increasing 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"],   label="Val Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curves"); axes[0].legend()

axes[1].plot(history["val_f1"],      label="Val F1")
axes[1].plot(history["val_bal_acc"], label="Val Balanced Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
axes[1].set_title("Validation Metrics"); axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
full_train_dataset = ConcatDataset([train_dataset, val_dataset])
full_train_loader  = DataLoader(full_train_dataset, batch_size=32, shuffle=True)

# Re-initialise model — important!
model = Baseline(dropout=0.5).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

full_history = {"train_loss": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for X, y in full_train_loader:
        X, y = X.to(device), y.to(device).float()
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(y)
    train_loss /= len(full_train_dataset)
    full_history["train_loss"].append(train_loss)
    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_loss:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(full_history["train_loss"], label="Train Loss (full data)")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Full Training Curve"); plt.legend()
plt.show()

In [ ]:
torch.save(model.state_dict(), "baseline_model.pth")

model_loaded = Baseline(dropout=0.5).to(device)
model_loaded.load_state_dict(torch.load("baseline_model.pth", map_location=device))
model_loaded.eval()
print("Model saved and reloaded successfully.")

In [ ]:
f1, bal_acc, auc, test_probs, test_labels = calculate_metrics(test_loader, model_loaded)
print(f"Test F1:            {f1:.4f}")
print(f"Test Balanced Acc:  {bal_acc:.4f}")
print(f"Test AUC:           {auc:.4f}")

In [ ]:
fpr, tpr, thresholds = roc_curve(test_labels, test_probs)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC Curve — Baseline CNN")
plt.legend()
plt.savefig("baseline_roc.png", dpi=150)
plt.show()

# --- Threshold selection ---
# In a clinical screening context, missing an OA case (false negative) is more
# costly than a false alarm. So we prioritise high sensitivity.
# Choose the threshold where sensitivity >= 0.90.
target_sensitivity = 0.90
idx = np.where(tpr >= target_sensitivity)[0][0]
chosen_threshold = thresholds[idx]
print(f"Chosen threshold: {chosen_threshold:.3f}  "
      f"(Sensitivity={tpr[idx]:.3f}, Specificity={1-fpr[idx]:.3f})")

In [ ]:
test_preds = (test_probs >= chosen_threshold).astype(int)
cm = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Healthy", "OA"])
disp.plot(cmap="Blues")
plt.title(f"Confusion Matrix (threshold={chosen_threshold:.2f})")
plt.savefig("baseline_confusion_matrix.png", dpi=150)
plt.show() 

In [ ]:
# Collect a few samples from the test loader
model_loaded.eval()
samples, labels_list, preds_list = [], [], []

with torch.no_grad():
    for X, y in test_loader:
        logits = model_loaded(X.to(device))
        probs  = torch.sigmoid(logits).cpu().numpy()
        preds  = (probs >= chosen_threshold).astype(int)
        for i in range(len(X)):
            samples.append(X[i])
            labels_list.append(int(y[i]))
            preds_list.append(preds[i])
        if len(samples) >= 8:
            break

class_names = ["Healthy", "OA"]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, img, gt, pred in zip(axes.flat, samples, labels_list, preds_list):
    # Denormalise for display (adapt denormalize() to your implementation)
    img_display = denormalize(img).squeeze().numpy()
    ax.imshow(img_display, cmap="gray")
    color = "green" if gt == pred else "red"
    ax.set_title(f"GT: {class_names[gt]}\nPred: {class_names[pred]}", color=color)
    ax.axis("off")
plt.suptitle("Test samples — Green: correct, Red: incorrect")
plt.tight_layout()
plt.savefig("baseline_test_samples.png", dpi=150)
plt.show()